# SimASM Python API Demo

This notebook demonstrates the SimASM **Python API** for interactive simulation and verification.

Using the Python API (with models as strings) **avoids the red squiggly linter warnings** that appear when using cell magics in Google Colab.

In [ ]:
# Install simasm (uncomment in Colab)
# !pip install simasm

import simasm
print(f"SimASM version: {simasm.__version__}")

## 1. Define Models

Define SimASM models as Python strings using `simasm.register_model()`.

This approach has **no linter warnings** because the DSL code is inside a Python string.

In [ ]:
# Event Graph M/M/5 Queue Model
mm5_eg_model = """
// =============================================================================
// Event Graph M/M/n Queue - SimASM v1.0
// =============================================================================
// A multi-server queueing system modeled using Event Graph formalism
// Based on Schruben (1983) Event Graph methodology
// =============================================================================

// =============================================================================
// IMPORT LIBRARIES
// =============================================================================

import Random as rnd
import Stdlib as lib

// =============================================================================
// DOMAIN DECLARATIONS
// =============================================================================

domain Object
domain Generator <: Object
domain Queue <: Object
domain Server <: Object
domain Load <: Object

// Event domain hierarchy
domain Event <: Object
domain ArriveEvent <: Event
domain AttemptToStartEvent <: Event
domain StartEvent <: Event
domain FinishEvent <: Event

// =============================================================================
// CONSTANT DECLARATIONS (Static 0-ary Functions)
// =============================================================================

const generator: Generator
const queue: Queue
const server: Server

const service_capacity: Nat
const iat_mean: Real
const ist_mean: Real
const sim_end_time: Real

// =============================================================================
// DYNAMIC VARIABLE DECLARATIONS (Dynamic 0-ary Functions)
// =============================================================================

var future_event_list: List<Event>
var sim_clocktime: Real
var load_id_counter: Nat
var current_event: Event

// =============================================================================
// RANDOM STREAM VARIABLES
// =============================================================================

var interarrival_time: rnd.exponential(iat_mean) as "arrivals"
var service_time: rnd.exponential(ist_mean) as "service"

// =============================================================================
// STATIC FUNCTION DECLARATIONS
// =============================================================================

static function id(obj: Object): Nat

// =============================================================================
// DYNAMIC FUNCTION DECLARATIONS
// =============================================================================

// Time counters
dynamic function arrival_time(l: Load): Real
dynamic function service_start_time(l: Load): Real
dynamic function departure_time(l: Load): Real

dynamic function total_time_in_system(s: Server): Real
dynamic function total_time_in_queue(s: Server): Real

// Entity lists
dynamic function arrivals(g: Generator): List<Load>
dynamic function queues(q: Queue): List<Load>
dynamic function serves(s: Server): List<Load>
dynamic function departures(s: Server): List<Load>

// Counters
dynamic function arrival_count(g: Generator): Nat
dynamic function queue_count(q: Queue): Nat
dynamic function service_count(s: Server): Nat
dynamic function departure_count(s: Server): Nat

// Event properties (stored as dynamic functions)
dynamic function event_rule(e: Event): Rule
dynamic function event_scheduled_time(e: Event): Real
dynamic function event_parameters(e: Event): List<Any>

// -----------------------------------------------------------------------------
// Observable State Derived Functions (for verification)
// These wrap the dynamic function accessors for cleaner predicate syntax
// -----------------------------------------------------------------------------

derived function get_queue_count(): Nat =
    queue_count(queue)

derived function get_servers_busy(): Nat =
    service_count(server)

derived function in_system(): Nat =
    get_queue_count() + get_servers_busy()

// Queue count predicates
derived function queue_eq_0(): Bool = get_queue_count() == 0
derived function queue_eq_1(): Bool = get_queue_count() == 1
derived function queue_eq_2(): Bool = get_queue_count() == 2
derived function queue_eq_3(): Bool = get_queue_count() == 3
derived function queue_eq_4(): Bool = get_queue_count() == 4
derived function queue_eq_5(): Bool = get_queue_count() == 5
derived function queue_ge_6(): Bool = get_queue_count() >= 6

// Servers busy predicates
derived function busy_eq_0(): Bool = get_servers_busy() == 0
derived function busy_eq_1(): Bool = get_servers_busy() == 1
derived function busy_eq_2(): Bool = get_servers_busy() == 2
derived function busy_eq_3(): Bool = get_servers_busy() == 3
derived function busy_eq_4(): Bool = get_servers_busy() == 4
derived function busy_eq_5(): Bool = get_servers_busy() == 5

// In-system predicates
derived function insys_eq_0(): Bool = in_system() == 0
derived function insys_eq_1(): Bool = in_system() == 1
derived function insys_eq_2(): Bool = in_system() == 2
derived function insys_eq_3(): Bool = in_system() == 3
derived function insys_eq_4(): Bool = in_system() == 4
derived function insys_eq_5(): Bool = in_system() == 5
derived function insys_ge_6(): Bool = in_system() >= 6

// =============================================================================
// RULES (Event Graph Routines)
// =============================================================================

// -----------------------------------------------------------------------------
// Initialization Routine
// -----------------------------------------------------------------------------

rule initialisation_routine() =
    // Create first arrival event (scheduled after first interarrival time)
    let first_event = new ArriveEvent
    event_rule(first_event) := "arrive"
    event_scheduled_time(first_event) := sim_clocktime + interarrival_time
    event_parameters(first_event) := []
    
    // Schedule to future event list
    lib.add(future_event_list, first_event)
endrule

// -----------------------------------------------------------------------------
// Timing Routine - advance clock and get next event
// -----------------------------------------------------------------------------

rule timing_routine() =
    // Sort FEL by scheduled time (uses dynamic function lookup)
    lib.sort(future_event_list, "event_scheduled_time")
    
    // Get next event
    if lib.length(future_event_list) > 0 then
        let next_event = lib.pop(future_event_list)
    
        // Advance clock
        sim_clocktime := event_scheduled_time(next_event)
        
        // Store current event for dispatch
        current_event := next_event
    endif
endrule

// -----------------------------------------------------------------------------
// Event Routine - dispatch to appropriate event handler
// -----------------------------------------------------------------------------

rule event_routine() =
    let r = event_rule(current_event)
    let params = event_parameters(current_event)
    lib.apply_rule(r, params)
endrule

// -----------------------------------------------------------------------------
// Run Routine - main simulation step
// -----------------------------------------------------------------------------

rule run_routine() =
    timing_routine()
    event_routine()
endrule

// -----------------------------------------------------------------------------
// Arrive Event Rule
// -----------------------------------------------------------------------------

rule arrive() =
    // Generate new load id
    load_id_counter := load_id_counter + 1
    
    // Create new load entity
    let load = new Load
    id(load) := load_id_counter
    
    // Record arrival
    arrival_time(load) := sim_clocktime
    lib.add(arrivals(generator), load)
    arrival_count(generator) := arrival_count(generator) + 1
    
    // Schedule AttemptToStart for this load (immediate)
    let attempt_event = new AttemptToStartEvent
    event_rule(attempt_event) := "attempt_to_start"
    event_scheduled_time(attempt_event) := sim_clocktime
    event_parameters(attempt_event) := [load]
    lib.add(future_event_list, attempt_event)
    
    // Schedule next arrival
    let next_arrive = new ArriveEvent
    event_rule(next_arrive) := "arrive"
    event_scheduled_time(next_arrive) := sim_clocktime + interarrival_time
    event_parameters(next_arrive) := []
    lib.add(future_event_list, next_arrive)
endrule

// -----------------------------------------------------------------------------
// Attempt To Start Event Rule
// -----------------------------------------------------------------------------

rule attempt_to_start(load: Load) =
    // Add load to queue
    lib.add(queues(queue), load)
    queue_count(queue) := queue_count(queue) + 1

    // Check if server capacity available and queue non-empty
    if lib.length(serves(server)) < service_capacity and lib.length(queues(queue)) > 0 then
        // Get first load in queue (will be removed in start rule)
        let load_to_start = lib.first(queues(queue))

        // Schedule Start event (immediate)
        let start_event = new StartEvent
        event_rule(start_event) := "start"
        event_scheduled_time(start_event) := sim_clocktime
        event_parameters(start_event) := [load_to_start]
        lib.add(future_event_list, start_event)
    endif
endrule

// -----------------------------------------------------------------------------
// Start Event Rule
// -----------------------------------------------------------------------------

rule start(load: Load) =
    // Remove load from queue
    lib.remove(queues(queue), load)
    queue_count(queue) := queue_count(queue) - 1

    // Record service start timestamp
    service_start_time(load) := sim_clocktime

    // Add load to service
    lib.add(serves(server), load)
    service_count(server) := service_count(server) + 1

    // Schedule Finish event
    let finish_event = new FinishEvent
    event_rule(finish_event) := "finish"
    event_scheduled_time(finish_event) := sim_clocktime + service_time
    event_parameters(finish_event) := [load]
    lib.add(future_event_list, finish_event)
endrule

// -----------------------------------------------------------------------------
// Finish Event Rule
// -----------------------------------------------------------------------------

rule finish(load: Load) =
    // Remove load from service
    lib.remove(serves(server), load)
    service_count(server) := service_count(server) - 1

    // Record departure and accumulate times
    departure_time(load) := sim_clocktime

    let time_in_system = sim_clocktime - arrival_time(load)
    let time_in_queue = service_start_time(load) - arrival_time(load)

    total_time_in_system(server) := total_time_in_system(server) + time_in_system
    total_time_in_queue(server) := total_time_in_queue(server) + time_in_queue
    
    // Add to departures
    lib.add(departures(server), load)
    departure_count(server) := departure_count(server) + 1
    
    // Check if more loads waiting (conditional scheduling edge)
    if lib.length(queues(queue)) > 0 then
        let next_load = lib.first(queues(queue))
        
        // Schedule Start for next load (immediate)
        let start_event = new StartEvent
        event_rule(start_event) := "start"
        event_scheduled_time(start_event) := sim_clocktime
        event_parameters(start_event) := [next_load]
        lib.add(future_event_list, start_event)
    endif
endrule

// =============================================================================
// INITIAL STATE
// =============================================================================

init:
    // Create constant objects
    generator := new Generator
    queue := new Queue
    server := new Server
    
    // System parameters
    service_capacity := 5
    iat_mean := 1.25
    ist_mean := 1.0
    
    // Time settings
    sim_clocktime := 0.0
    sim_end_time := 1000.0
    
    // Counters
    load_id_counter := 0
    arrival_count(generator) := 0
    queue_count(queue) := 0
    service_count(server) := 0
    departure_count(server) := 0
    
    // Lists
    future_event_list := []
    arrivals(generator) := []
    queues(queue) := []
    serves(server) := []
    departures(server) := []

    // Time statistics
    total_time_in_system(server) := 0.0
    total_time_in_queue(server) := 0.0
endinit

// =============================================================================
// MAIN RULE
// =============================================================================

main rule main =
    // Initialize on first step
    if sim_clocktime == 0.0 and lib.length(future_event_list) == 0 then
        initialisation_routine()
    endif
    
    // Run while events exist and time not exceeded
    if lib.length(future_event_list) > 0 and sim_clocktime < sim_end_time then
        run_routine()
    endif
endrule
"""

# Register the Event Graph model
simasm.register_model("mm5_eg", mm5_eg_model)
print("Registered model: mm5_eg")

In [ ]:
# Activity Cycle Diagram M/M/5 Queue Model
mm5_acd_model = """
// =============================================================================
// Activity Cycle Diagram M/M/n Queue - SimASM v1.0
// =============================================================================
// A multi-server queueing system modeled using ACD formalism
// Based on Tocher (1960), Carrie (1988)
// Three-phase approach: A-phase (time advance), B-phase (scan), C-phase (execute)
// =============================================================================

// =============================================================================
// IMPORT LIBRARIES
// =============================================================================

import Random as rnd
import Stdlib as lib

// =============================================================================
// DOMAIN DECLARATIONS
// =============================================================================

domain Object

// Passive states (Queues - ovals in ACD)
domain Queue <: Object

// Active states (Activities - rectangles in ACD)
domain Activity <: Object

// Tokens (entities flowing through the diagram)
domain Token <: Object
domain Job <: Token
domain Resource <: Token

// Bound-to-occur events
domain BTOEvent <: Object

// =============================================================================
// QUEUE DECLARATIONS (Passive States)
// =============================================================================

const C: Queue                  // Creator resource queue
const Q: Queue                  // Job waiting queue
const S: Queue                  // Server resource queue
const Jobs: Queue               // Completed jobs (sink)

// =============================================================================
// ACTIVITY DECLARATIONS (Active States)
// =============================================================================

const Create: Activity          // Job creation activity
const Serve: Activity           // Service activity

// =============================================================================
// CONSTANT DECLARATIONS
// =============================================================================

const iat_mean: Real            // Inter-arrival time mean
const ist_mean: Real            // Service time mean
const num_servers: Nat          // n in M/M/n
const sim_end_time: Real

// =============================================================================
// DYNAMIC VARIABLE DECLARATIONS
// =============================================================================

var future_event_list: List<BTOEvent>
var sim_clocktime: Real
var job_id_counter: Nat
var activities_started: Bool
var current_phase: String
var total_sojourn_time: Real
var total_time_in_queue: Real
var departure_count: Nat

// =============================================================================
// RANDOM STREAM VARIABLES
// =============================================================================

var duration_create: rnd.exponential(iat_mean) as "arrivals"
var duration_serve: rnd.exponential(ist_mean) as "service"

// =============================================================================
// DYNAMIC FUNCTION DECLARATIONS
// =============================================================================

// Marking (token counts in queues)
dynamic function marking(q: Queue): Nat
dynamic function tokens(q: Queue): List<Token>

// BTO-Event properties
dynamic function bto_activity(e: BTOEvent): Activity
dynamic function bto_scheduled_time(e: BTOEvent): Real
dynamic function bto_tokens(e: BTOEvent): List<Token>

// statistics collector
dynamic function arrival_time(j: Job): Real
dynamic function service_start_time(j: Job): Real

// Token identification
static function id(t: Token): Nat

// -----------------------------------------------------------------------------
// Observable State (aligned with Event Graph)
// -----------------------------------------------------------------------------

derived function queue_count(): Nat =
    marking(Q)

derived function servers_busy(): Nat =
    num_servers - marking(S)

derived function in_system(): Nat =
    queue_count() + servers_busy()

// -----------------------------------------------------------------------------
// Observable State Predicates (for verification)
// -----------------------------------------------------------------------------

// Queue count predicates
derived function queue_eq_0(): Bool = queue_count() == 0
derived function queue_eq_1(): Bool = queue_count() == 1
derived function queue_eq_2(): Bool = queue_count() == 2
derived function queue_eq_3(): Bool = queue_count() == 3
derived function queue_eq_4(): Bool = queue_count() == 4
derived function queue_eq_5(): Bool = queue_count() == 5
derived function queue_ge_6(): Bool = queue_count() >= 6

// Servers busy predicates
derived function busy_eq_0(): Bool = servers_busy() == 0
derived function busy_eq_1(): Bool = servers_busy() == 1
derived function busy_eq_2(): Bool = servers_busy() == 2
derived function busy_eq_3(): Bool = servers_busy() == 3
derived function busy_eq_4(): Bool = servers_busy() == 4
derived function busy_eq_5(): Bool = servers_busy() == 5

// In-system predicates
derived function insys_eq_0(): Bool = in_system() == 0
derived function insys_eq_1(): Bool = in_system() == 1
derived function insys_eq_2(): Bool = in_system() == 2
derived function insys_eq_3(): Bool = in_system() == 3
derived function insys_eq_4(): Bool = in_system() == 4
derived function insys_eq_5(): Bool = in_system() == 5
derived function insys_ge_6(): Bool = in_system() >= 6

// =============================================================================
// AT-BEGIN CONDITIONS (Activity start preconditions)
// =============================================================================

derived function at_begin_condition_create(): Bool =
    marking(C) > 0

derived function at_begin_condition_serve(): Bool =
    marking(S) > 0 and marking(Q) > 0

// =============================================================================
// RULES (ACD Phases)
// =============================================================================

// -----------------------------------------------------------------------------
// Phase 0: Initialization
// -----------------------------------------------------------------------------

rule initialisation_routine() =
    // Initialize creator token
    let creator = new Resource
    id(creator) := 0
    tokens(C) := [creator]

    // Initialize server tokens
    tokens(S) := []
    init_servers(num_servers)
endrule

// Helper rule to initialize server tokens
rule init_servers(remaining: Nat) =
    if remaining > 0 then
        let server_token = new Resource
        id(server_token) := num_servers - remaining + 1
        lib.add(tokens(S), server_token)
        init_servers(remaining - 1)
    endif
endrule

// -----------------------------------------------------------------------------
// At-begin Actions (Activity instantiation)
// -----------------------------------------------------------------------------

rule at_begin_action_create() =
    // Consume creator token from C
    marking(C) := marking(C) - 1
    let creator_token = lib.pop(tokens(C))
    
    // Schedule BTO-event for activity completion
    let bto_event = new BTOEvent
    bto_activity(bto_event) := Create
    bto_scheduled_time(bto_event) := sim_clocktime + duration_create
    bto_tokens(bto_event) := [creator_token]
    lib.add(future_event_list, bto_event)
endrule

rule at_begin_action_serve() =
    // Consume server token from S
    marking(S) := marking(S) - 1
    let server_token = lib.pop(tokens(S))
    
    // Consume job token from Q
    marking(Q) := marking(Q) - 1
    let job_token = lib.pop(tokens(Q))

    // Record when job leaves queue (starts service)
    service_start_time(job_token) := sim_clocktime
    
    // Schedule BTO-event for activity completion
    let bto_event = new BTOEvent
    bto_activity(bto_event) := Serve
    bto_scheduled_time(bto_event) := sim_clocktime + duration_serve
    bto_tokens(bto_event) := [server_token, job_token]
    lib.add(future_event_list, bto_event)
endrule

// -----------------------------------------------------------------------------
// At-end Actions (Activity completion)
// -----------------------------------------------------------------------------

rule at_end_action_create(bound_tokens: List<Token>) =
    // Arc 1: Return creator resource to C
    let creator_token = lib.get(bound_tokens, 0)
    lib.add(tokens(C), creator_token)
    marking(C) := marking(C) + 1
    
    // Arc 2: Create new job and add to Q
    job_id_counter := job_id_counter + 1
    let new_job = new Job
    id(new_job) := job_id_counter
    arrival_time(new_job) := sim_clocktime
    lib.add(tokens(Q), new_job)
    marking(Q) := marking(Q) + 1
endrule

rule at_end_action_serve(bound_tokens: List<Token>) =
    // Arc 1: Return server resource to S
    let server_token = lib.get(bound_tokens, 0)
    lib.add(tokens(S), server_token)
    marking(S) := marking(S) + 1
    
    // Arc 2: Move completed job to Jobs (sink)
    let job_token = lib.get(bound_tokens, 1)

    // Accumulate time statistics
    let time_in_system = sim_clocktime - arrival_time(job_token)
    let time_in_queue = service_start_time(job_token) - arrival_time(job_token)
    
    total_sojourn_time := total_sojourn_time + time_in_system
    total_time_in_queue := total_time_in_queue + time_in_queue

    departure_count := departure_count + 1
    
    lib.add(tokens(Jobs), job_token)
    marking(Jobs) := marking(Jobs) + 1
endrule

// -----------------------------------------------------------------------------
// Phase 1: B-phase (Scanning Phase) - One activity per step
// -----------------------------------------------------------------------------

rule scanning_phase_single() =
    activities_started := false

    // Try Create first
    if not activities_started and at_begin_condition_create() then
        at_begin_action_create()
        activities_started := true
    endif

    // Try ONE Serve only if Create didn't start
    if not activities_started and at_begin_condition_serve() then
        at_begin_action_serve()
        activities_started := true
    endif
endrule

// -----------------------------------------------------------------------------
// Phase 2: A-phase (Timing Phase) - advance clock
// -----------------------------------------------------------------------------

rule timing_phase() =
    // Sort FEL by scheduled time (uses dynamic function lookup)
    lib.sort(future_event_list, "bto_scheduled_time")
    
    // Get next event
    let next_event = lib.pop(future_event_list)
    
    // Advance clock
    sim_clocktime := bto_scheduled_time(next_event)
    
    // Execute the event
    executing_phase(next_event)
endrule

// -----------------------------------------------------------------------------
// Phase 3: C-phase (Executing Phase)
// -----------------------------------------------------------------------------

rule executing_phase(bto_event: BTOEvent) =
    let activity = bto_activity(bto_event)
    let bound_tokens = bto_tokens(bto_event)
    
    if activity == Create then
        at_end_action_create(bound_tokens)
    endif
    
    if activity == Serve then
        at_end_action_serve(bound_tokens)
    endif
endrule

// =============================================================================
// INITIAL STATE
// =============================================================================

init:
    // Create constant objects (Queues and Activities)
    C := new Queue
    Q := new Queue
    S := new Queue
    Jobs := new Queue
    Create := new Activity
    Serve := new Activity

    // M/M/n configuration (must be set before markings that depend on it)
    num_servers := 5

    // Time parameters (rates as means: mean = 1/rate)
    iat_mean := 1.25
    ist_mean := 1.0

    // Initialize markings (matching EG initial state: 0 busy, 0 in queue)
    marking(C) := 1
    marking(Q) := 0
    marking(S) := num_servers
    marking(Jobs) := 0

    // Initialize token lists
    tokens(C) := []
    tokens(Q) := []
    tokens(S) := []
    tokens(Jobs) := []
    
    // Simulation bounds
    sim_clocktime := 0.0
    sim_end_time := 1000.0
    total_sojourn_time := 0.0
    total_time_in_queue := 0.0
    
    // Initialize FEL
    future_event_list := []
    
    // Job counter
    job_id_counter := 0
    departure_count := 0
    
    // Scanning flag and phase control
    activities_started := false
    current_phase := "scan"
endinit

// =============================================================================
// MAIN RULE
// =============================================================================

main rule main =
    // Initialize tokens on first step (tokens(C) is empty until initialisation_routine runs)
    if sim_clocktime == 0.0 and lib.length(tokens(C)) == 0 then
        initialisation_routine()
        current_phase := "scan"
    endif

    // Run while time not exceeded
    if sim_clocktime < sim_end_time then
        if current_phase == "scan" then
            scanning_phase_single()
            // Switch to timing only if no activity could start
            if not activities_started then
                current_phase := "time"
            endif
        else
            // current_phase == "time"
            if lib.length(future_event_list) > 0 then
                timing_phase()
            endif
            current_phase := "scan"
        endif
    endif
endrule
"""

# Register the ACD model
simasm.register_model("mm5_acd", mm5_acd_model)
print("Registered model: mm5_acd")

In [ ]:
# List registered models
print("Registered models:", simasm.list_models())

## 2. Run Experiments

Use `simasm.run_experiment()` with an experiment specification string.

In [ ]:
# Run an experiment using the Python API
experiment_spec = """
experiment LittlesLawEG:
    model := "mm5_eg"
    
    replication:
        count: 10
        warm_up_time: 100.0
        run_length: 1000.0
        seed_strategy: "incremental"
        base_seed: 12345
    endreplication
    
    statistics:
        // L: Average number in system
        stat L_system: time_average
            expression: "lib.length(queues(queue)) + lib.length(serves(server))"
        endstat
        
        // L_q: Average number in queue
        stat L_queue: time_average
            expression: "lib.length(queues(queue))"
        endstat
        
        // L_s: Average number in service
        stat L_service: time_average
            expression: "lib.length(serves(server))"
        endstat

        // W: Average time in system
        stat W_system: count
            expression: "total_time_in_system(server) / departure_count(server)"
        endstat

        // W_q: Average time in queue
        stat W_queue: count
            expression: "total_time_in_queue(server) / departure_count(server)"
        endstat
        
        // Total arrivals
        stat total_arrivals: count
            expression: "arrival_count(generator)"
        endstat
        
        // Server utilization
        stat rho_utilization: utilization
            expression: "lib.length(serves(server)) > 0"
        endstat
        
    endstatistics
    
    output:
        format: "json"
        file_path: "littles_law_results.json"
    endoutput
endexperiment
"""

result = simasm.run_experiment(experiment_spec)

# Display results nicely
simasm.display_experiment_result(result)

## 3. Run Verification

Use `simasm.verify()` to verify stutter equivalence between models.

In [ ]:
# Run verification using the Python API
verification_spec = """
verification MM5_W_Stutter_Equivalence:
    models:
        import EG from "mm5_eg"
        import ACD from "mm5_acd"
    endmodels

    seed: 42

    labels:
        // Servers busy predicates (observation level W)
        label busy_eq_0 for EG: "busy_eq_0()"
        label busy_eq_0 for ACD: "busy_eq_0()"

        label busy_eq_1 for EG: "busy_eq_1()"
        label busy_eq_1 for ACD: "busy_eq_1()"

        label busy_eq_2 for EG: "busy_eq_2()"
        label busy_eq_2 for ACD: "busy_eq_2()"

        label busy_eq_3 for EG: "busy_eq_3()"
        label busy_eq_3 for ACD: "busy_eq_3()"

        label busy_eq_4 for EG: "busy_eq_4()"
        label busy_eq_4 for ACD: "busy_eq_4()"

        label busy_eq_5 for EG: "busy_eq_5()"
        label busy_eq_5 for ACD: "busy_eq_5()"

    endlabels

    observables:
        observable busy_eq_0:
            EG -> busy_eq_0
            ACD -> busy_eq_0
        endobservable

        observable busy_eq_1:
            EG -> busy_eq_1
            ACD -> busy_eq_1
        endobservable

        observable busy_eq_2:
            EG -> busy_eq_2
            ACD -> busy_eq_2
        endobservable

        observable busy_eq_3:
            EG -> busy_eq_3
            ACD -> busy_eq_3
        endobservable

        observable busy_eq_4:
            EG -> busy_eq_4
            ACD -> busy_eq_4
        endobservable

        observable busy_eq_5:
            EG -> busy_eq_5
            ACD -> busy_eq_5
        endobservable

    endobservables

    check:
        type: stutter_equivalence
        run_length: 1000.0
        timeout: 600
    endcheck

    output:
        format: "csv"
        file_path: "verification_results.csv"
        include_counterexample: true
    endoutput

endverification
"""

result = simasm.verify(verification_spec)

# Display results nicely
simasm.display_verification_result(result)

In [ ]:
# Clear all registered models
simasm.clear_models()
print("Cleared all models.")

## Summary

The Python API provides the same functionality as cell magics, but with:

1. **No linter warnings** - Models are defined as Python strings
2. **Better IDE support** - Works well with code completion and navigation
3. **Programmatic control** - Easier to loop over experiments or parameters

| Cell Magic | Python API |
|------------|------------|
| `%%simasm model --name X` | `simasm.register_model("X", source)` |
| `%%simasm experiment` | `simasm.run_experiment(spec)` |
| `%%simasm verify` | `simasm.verify(spec)` |
| `%simasm_models` | `simasm.list_models()` |
| `%simasm_clear` | `simasm.clear_models()` |